# Stage 3 - DPO Preference Alignment

**Goal:** push the SFT model to prefer *correct, specific* answers over *wrong / generic* ones
(hallucinated tables, missing DISTINCT, lazy SELECT *, wrong joins).

Data: `ecomm-db-preference` on the Hugging Face Hub (63 `{prompt, chosen, rejected}` examples).

In [ ]:
# Install Unsloth (Colab has a compatible CUDA GPU). Restart runtime if prompted.
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Runtime > Change runtime type > T4 GPU")

## 1. Config

In [ ]:
# --- Datasets live on the Hugging Face Hub (pushed via scripts/push_to_hf.py) ---
HF_USER = "Rajesh507"   # <-- your Hugging Face username
DS_NONINSTRUCT = f"{HF_USER}/ecomm-db-noninstruct"
DS_INSTRUCTION = f"{HF_USER}/ecomm-db-instruction"
DS_PREFERENCE  = f"{HF_USER}/ecomm-db-preference"

# If the datasets are PRIVATE, log in first (needs a read token):
# from huggingface_hub import login; login()

In [ ]:
# Mount Google Drive ONLY to persist trained adapters across sessions.
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/Ecomm-ai-assistant-finetuning/outputs'
os.makedirs(SAVE_DIR, exist_ok=True)
print("Adapters will be saved to:", SAVE_DIR)

## 2. Load the SFT model (Stage 2 adapter from Drive)

In [ ]:
MODEL_NAME = "unsloth/Qwen2.5-Coder-1.5B"   # Coder variant is stronger at SQL. Alt: "unsloth/Qwen2.5-1.5B"
MAX_SEQ_LEN = 2048

In [ ]:
from unsloth import FastLanguageModel
SFT_ADAPTER = os.path.join(SAVE_DIR, 'stage2_sft_adapter')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = SFT_ADAPTER,
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,
    load_in_4bit = True,
)

In [ ]:
from unsloth import FastLanguageModel
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none",
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 3. Load and format the preference dataset

In [ ]:
# A single, consistent prompt template used across ALL three stages.
PROMPT = """Below is a question about the client e-commerce database schema. \
Write a response that correctly answers it, giving the exact table name(s) or a valid SQL query.

### Question:
{}

### Answer:
{}"""

In [ ]:
from datasets import load_dataset
EOS = tokenizer.eos_token

def format_pref(ex):
    return {
        "prompt": PROMPT.format(ex["prompt"], ""),
        "chosen": ex["chosen"] + EOS,
        "rejected": ex["rejected"] + EOS,
    }

ds = load_dataset(DS_PREFERENCE, split="train").map(format_pref)
print(ds); print(ds[0])

## 4. Configure and run DPO

In [ ]:
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,          # LoRA -> no separate reference model needed (saves memory)
    tokenizer = tokenizer,
    train_dataset = ds,
    args = DPOConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,
        learning_rate = 5e-5,
        beta = 0.1,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.0,
        lr_scheduler_type = "linear",
        seed = 3407,
        max_length = MAX_SEQ_LEN,
        max_prompt_length = 512,
        output_dir = "outputs_stage3",
        report_to = "none",
    ),
)
dpo_trainer.train()

## 5. Save the DPO-aligned model

In [ ]:
dpo_path = os.path.join(SAVE_DIR, 'stage3_dpo_adapter')
model.save_pretrained(dpo_path)
tokenizer.save_pretrained(dpo_path)
print("Saved to", dpo_path)

# Optional: merge to a standalone 16-bit model (no adapter needed to load)
# model.save_pretrained_merged(os.path.join(SAVE_DIR,'final_merged_16bit'), tokenizer, save_method='merged_16bit')

## 6. Test after DPO

In [ ]:
FastLanguageModel.for_inference(model)

def ask(question, max_new_tokens=128):
    text = PROMPT.format(question, "")
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.2, do_sample=False)
    return tokenizer.decode(out[0], skip_special_tokens=True).split("### Answer:")[-1].strip()

for q in [
    "Find the number of unique orders placed by customer 1001.",
    "Which table stores product images?",
    "Find the top 5 customers by total spend.",
]:
    print("Q:", q); print("A:", ask(q)); print("-"*60)

### Done
Three adapters are now in Drive (Stage 1 / 2 / 3). Use `src/inference.py` for the final model and fill in the `reports/` comparison tables.